In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5000/'
rdm_api_url = 'http://localhost:8000/v2/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'TEST-RDMFS-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
minio_enabled = False
minio_service_name = 'MinIO (CI)'
minio_access_key = None
minio_secret_key = None
minio_bucket = None
read_credentials_from_environment = False
rdmfs_image = 'rdmfs:e2e'
rdmfs_container_name = 'rdmfs-e2e'
rdmfs_mount_path = '/mnt/test'
default_result_path = None
close_on_fail = False
transition_timeout = 60000
sync_timeout = 60
sync_poll_interval = 2


In [ ]:
import os
import tempfile

if isinstance(minio_enabled, str):
    assert minio_enabled in ['true', 'false'], minio_enabled
    minio_enabled = minio_enabled.lower() == 'true'
if isinstance(read_credentials_from_environment, str):
    assert read_credentials_from_environment in ['true', 'false'], (
        read_credentials_from_environment
    )
    read_credentials_from_environment = (
        read_credentials_from_environment.lower() == 'true'
    )

if read_credentials_from_environment:
    idp_username_1 = os.environ['RDMFS_E2E_USERNAME']
    idp_password_1 = os.environ['RDMFS_E2E_PASSWORD']
    if minio_enabled:
        minio_access_key = os.environ['MINIO_ACCESS_KEY']
        minio_secret_key = os.environ['MINIO_SECRET_KEY']
elif idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')

if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if minio_enabled and minio_access_key is None:
    minio_access_key = input(prompt='MinIO access key')
if minio_enabled and minio_secret_key is None:
    minio_secret_key = getpass(prompt='MinIO secret key')
if minio_enabled and minio_bucket is None:
    minio_bucket = input(prompt='MinIO bucket')

work_dir = tempfile.mkdtemp(prefix='rdmfs-e2e-')
if default_result_path is None:
    default_result_path = work_dir
os.makedirs(default_result_path, exist_ok=True)


In [ ]:
assert target_storage_id in ['osfstorage', 's3compatsigv4']
assert (target_storage_id == 's3compatsigv4') == minio_enabled
assert sync_timeout < 180, 'The test must not wait out the current list-cache TTL'


# RDMFS E2E テスト [ストレージ双方向操作]

- サブシステム名: rdmfs / ファイル
- ページ/アドオン: NII Storage または S3 Compatible Storage (SigV4)
- 機能分類: GUI と FUSE の双方向整合性
- シナリオ名: 同一プロジェクト・同一マウントを使った相互更新
- 用意するテストデータ: プロジェクトを持たない既存ユーザー、RDM URL、MinIO 接続情報（SigV4 の場合）

対象プロジェクトと Personal Access Token は、この Notebook がブラウザ GUI から作成・削除する。rdmfs はリポジトリ同梱 Dockerfile の標準 `CMD` で起動し、再マウントせずに最後まで使用する。


In [ ]:
import importlib
import pathlib
import re
import shutil
import subprocess
import sys
import time
from urllib.parse import urlparse

resources_dir = os.path.abspath('resources')
if resources_dir not in sys.path:
    sys.path.insert(0, resources_dir)

import scripts.grdm
import scripts.playwright
importlib.reload(scripts.grdm)
importlib.reload(scripts.playwright)

from playwright.async_api import TimeoutError as PlaywrightTimeoutError
from playwright.async_api import expect
from scripts import grdm
from scripts.playwright import finish_pw_context, init_pw_context, run_pw

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


def run_command(arguments, check=True):
    result = subprocess.run(arguments, capture_output=True, text=True)
    if check and result.returncode != 0:
        safe_arguments = [
            'RDM_TOKEN=<redacted>'
            if str(value).startswith('RDM_TOKEN=') else value
            for value in arguments
        ]
        raise RuntimeError(
            f'Command failed ({result.returncode}): {safe_arguments!r}\n'
            f'stdout:\n{result.stdout}\nstderr:\n{result.stderr}'
        )
    return result


def docker_exec(*arguments, check=True):
    return run_command(
        ['docker', 'exec', rdmfs_container_name, *[str(value) for value in arguments]],
        check=check,
    )


def rdmfs_container_id():
    return run_command(
        ['docker', 'inspect', '--format', '{{.Id}}', rdmfs_container_name]
    ).stdout.strip()


def wait_for_rdmfs(predicate, description):
    deadline = time.monotonic() + sync_timeout
    while time.monotonic() < deadline:
        if predicate():
            return
        time.sleep(sync_poll_interval)
    logs = run_command(['docker', 'logs', rdmfs_container_name], check=False)
    raise AssertionError(
        f'Timed out waiting for {description}.\n'
        f'rdmfs stdout:\n{logs.stdout}\nrdmfs stderr:\n{logs.stderr}'
    )


async def expand_folder(page, folder_name, timeout):
    folder = grdm.get_select_folder_title_locator(page, folder_name)
    try:
        await folder.wait_for(state='visible', timeout=timeout)
    except PlaywrightTimeoutError:
        return False
    collapsed = grdm.get_select_folder_toggle_locator(page, folder_name, collapsed=True)
    if await collapsed.is_visible():
        await collapsed.click()
        expanded = grdm.get_select_folder_toggle_locator(
            page, folder_name, expanded=True
        )
        try:
            await expanded.wait_for(state='visible', timeout=timeout)
        except PlaywrightTimeoutError:
            return False
    return True


async def wait_for_gui_entries(page, folder_name, present_names=(), absent_names=()):
    deadline = time.monotonic() + sync_timeout
    while time.monotonic() < deadline:
        await page.reload(wait_until='domcontentloaded')
        remaining = max(0, deadline - time.monotonic())
        attempt_timeout = min(sync_poll_interval, remaining) * 1000
        storage = grdm.get_select_expanded_storage_title_locator(page, target_storage_name)
        try:
            await storage.wait_for(state='visible', timeout=attempt_timeout)
        except PlaywrightTimeoutError:
            continue
        if not await expand_folder(page, folder_name, attempt_timeout):
            continue

        try:
            for name in present_names:
                await grdm.get_select_file_title_locator(page, name).wait_for(
                    state='visible', timeout=attempt_timeout
                )
        except PlaywrightTimeoutError:
            continue

        absent = [
            not await grdm.get_select_file_title_locator(page, name).is_visible()
            for name in absent_names
        ]
        if all(absent):
            return
    raise AssertionError(
        f'Timed out waiting for GUI entries: folder={folder_name!r}, '
        f'present={present_names!r}, absent={absent_names!r}'
    )


project_id = None
project_url = None
rdm_token = None
rdmfs_initial_container_id = None
token_name = f'rdmfs-e2e-{datetime.now().strftime("%Y%m%d-%H%M%S")}'


## ブラウザで GakuNin RDM にログインする

FakeCAS の既存ユーザーでログインし、プロジェクトを持たないダッシュボードが表示されること。


In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1,
        transition_timeout=transition_timeout,
    )
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-dashboard-item-title]')).to_have_count(0)

await run_pw(_step)


## GUI から試験対象プロジェクトを作成して開く

「新規プロジェクト作成」から指定名のプロジェクトを作成し、ダッシュボードから開けること。URL から、後続の rdmfs マウントに使用する project ID を取得する。


In [ ]:
async def _step(page):
    created = await grdm.ensure_project_exists(
        page, rdm_project_name, transition_timeout=transition_timeout
    )
    assert created, 'The target project must be created by this Notebook'
    await page.locator(
        f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]'
    ).click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(
        timeout=transition_timeout
    )

    global project_id, project_url
    project_id = urlparse(page.url).path.strip('/').split('/')[0]
    assert re.fullmatch(r'[a-z0-9]{5}', project_id), project_id
    project_url = f'{rdm_url.rstrip("/")}/{project_id}/'

await run_pw(_step)


## 対象ストレージを GUI で準備する

Default Storage の場合は NII Storage をそのまま使用する。MinIO の場合は S3 Compatible Storage (SigV4) を有効化し、GUI から MinIO アカウントとバケットを接続する。最後にファイル画面で対象ストレージが表示されること。


In [ ]:
async def _step(page):
    if minio_enabled:
        await grdm.enable_addon(
            page, target_storage_name, transition_timeout=transition_timeout
        )
        connect_link = page.locator(
            f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]'
            '/..//a[contains(text(), "アカウントに接続する")]'
        )
        await expect(connect_link).to_be_visible(timeout=transition_timeout)
        await connect_link.click()

        credentials = page.locator(f'#{target_storage_id}InputCredentials')
        await expect(credentials.locator('.btn-success')).to_be_enabled(
            timeout=transition_timeout
        )
        await credentials.locator('select#selected_service').select_option(
            minio_service_name
        )
        await credentials.locator('input[name="access_key"]').fill(minio_access_key)
        await credentials.locator('input[name="secret_key"]').fill(minio_secret_key)
        await credentials.locator('.btn-success').click()

        connect_button = page.get_by_role('button', name='接続', exact=True)
        await expect(connect_button).to_be_enabled(timeout=transition_timeout)
        await connect_button.click()
        bucket_radio = page.locator(
            f'//span[text() = "{minio_bucket}"]/../../..//input[@type = "radio"]'
        )
        await expect(bucket_radio).to_be_visible(timeout=transition_timeout)
        await bucket_radio.click()
        save_button = page.locator(
            f'//*[@class = "{target_storage_id}-confirm-selection"]'
            '//input[@value = "保存"]'
        )
        await expect(save_button).to_be_enabled(timeout=transition_timeout)
        await save_button.click()
        linked_message = page.locator(
            f'//*[@id = "{target_storage_id}Scope"]'
            '/*[contains(@class, "help-block")]'
            '//*[contains(text(), "正常にリンクされました")]'
        )
        await expect(linked_message).to_be_visible(timeout=transition_timeout)

    await page.goto(f'{project_url}files/')
    await expect(
        grdm.get_select_expanded_storage_title_locator(page, target_storage_name)
    ).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## GUI から Personal Access Token を作成する

ユーザー設定の Personal Access Token 画面で `osf.full_write` scope のトークンを作成し、表示された値を rdmfs 起動用に保持する。PAT の操作は証跡記録対象外のプライベートブラウザで行い、記録対象のブラウザはプロジェクトのファイル画面を維持して、証跡にトークンを残さないこと。


In [ ]:
async def _step(page):
    global rdm_token

    browser = page.context.browser
    assert browser is not None
    unrecorded_context = await browser.new_context(
        locale='ja-JP',
        storage_state=await page.context.storage_state(),
    )
    unrecorded_page = await unrecorded_context.new_page()
    try:
        await unrecorded_page.goto(f'{project_url}files/')
        auth_toggle = unrecorded_page.locator(
            'a[aria-label="Toggle auth dropdown"]'
        )
        await expect(auth_toggle).to_be_visible(timeout=transition_timeout)
        await auth_toggle.click()

        settings_link = unrecorded_page.locator(
            'ul.auth-dropdown a[href="/settings/"]'
        )
        await expect(settings_link).to_be_visible(timeout=transition_timeout)
        await settings_link.click()

        personal_tokens_link = unrecorded_page.locator(
            'a[href="/settings/tokens/"]'
        )
        await expect(personal_tokens_link).to_be_visible(
            timeout=transition_timeout
        )
        await personal_tokens_link.click()

        create_link = unrecorded_page.locator(
            'a[href="/settings/tokens/create/"]'
        )
        await expect(create_link).to_be_visible(timeout=transition_timeout)
        await create_link.click()

        await unrecorded_page.locator('#token-fields input[type="text"]').fill(
            token_name
        )
        await unrecorded_page.locator('input[id="osf.full_write"]').check()
        await unrecorded_page.locator(
            '#token-fields button[type="submit"]:visible'
        ).click()
        token_value = unrecorded_page.locator('#token-keys samp')
        await expect(token_value).not_to_be_empty(timeout=transition_timeout)
        rdm_token = (await token_value.text_content()).strip()
        assert len(rdm_token) >= 20

        await unrecorded_page.goto(f'{rdm_url.rstrip("/")}/settings/tokens/')
        token_row = unrecorded_page.locator(
            f'//tr[.//span[text() = "{token_name}"]]'
        )
        await expect(token_row).to_be_visible(timeout=transition_timeout)
    finally:
        await unrecorded_context.close()

    await expect(
        grdm.get_select_expanded_storage_title_locator(page, target_storage_name)
    ).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 同梱 Dockerfile の標準構成で rdmfs を起動し、FUSE の一覧キャッシュを成立させる

`DEV=false` の通常ビルド済みイメージを、ソース bind mount やコマンド上書きなしで privileged・host network により起動する。対象ストレージがマウントされるまでポーリングし、最初の `ls` で FUSE 側の一覧状態を読み込む。


In [ ]:
run_command([
    'docker', 'run', '--detach',
    '--name', rdmfs_container_name,
    '--privileged',
    '--network', 'host',
    '--env', f'RDM_NODE_ID={project_id}',
    '--env', f'RDM_TOKEN={rdm_token}',
    '--env', f'RDM_API_URL={rdm_api_url}',
    '--env', f'MOUNT_PATH={rdmfs_mount_path}',
    rdmfs_image,
])

storage_path = f'{rdmfs_mount_path}/{target_storage_id}'
wait_for_rdmfs(
    lambda: docker_exec('test', '-d', storage_path, check=False).returncode == 0,
    f'FUSE storage mount {storage_path}',
)
rdmfs_initial_container_id = rdmfs_container_id()
initial_listing = docker_exec('ls', '-1', storage_path).stdout.splitlines()
assert initial_listing == [], initial_listing


## GUI でファイルをアップロードし、同じ rdmfs マウントから参照する

FUSE 側で空の一覧を読んだ後、GUI からファイルをアップロードする。rdmfs を再起動・再マウントせず、明示した同期タイムアウト内にファイルが参照可能となり、内容が一致すること。


In [ ]:
gui_filename = 'GUIから作成.txt'
gui_content = 'created through the GakuNin RDM GUI\n'
gui_filepath = pathlib.Path(work_dir) / gui_filename
gui_filepath.write_text(gui_content, encoding='utf-8')

async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(gui_filepath))
    await grdm.wait_for_uploaded(page, gui_filename, timeout=transition_timeout)

await run_pw(_step)

gui_rdmfs_path = f'{storage_path}/{gui_filename}'

def gui_file_reached_rdmfs():
    result = docker_exec('cat', gui_rdmfs_path, check=False)
    return result.returncode == 0 and result.stdout == gui_content

wait_for_rdmfs(gui_file_reached_rdmfs, f'{gui_filename} uploaded through the GUI')
assert rdmfs_container_id() == rdmfs_initial_container_id


## rdmfs でフォルダと日本語名ファイルを作成し、GUI から参照する

GUI 側でも対象ストレージの一覧が表示済みの状態から、同じ FUSE マウントでフォルダとファイルを作成する。GUI を通常の再読込でポーリングし、フォルダとファイルがツリーに現れること。


In [ ]:
rdmfs_folder = 'rdmfs-folder'
rdmfs_filename = '日本語-rdmfs.txt'
rdmfs_content = 'created through rdmfs\n'
rdmfs_folder_path = f'{storage_path}/{rdmfs_folder}'
rdmfs_file_path = f'{rdmfs_folder_path}/{rdmfs_filename}'

docker_exec('mkdir', rdmfs_folder_path)
docker_exec(
    'python3', '-c',
    'from pathlib import Path; import sys; Path(sys.argv[1]).write_text(sys.argv[2], encoding="utf-8")',
    rdmfs_file_path, rdmfs_content,
)

async def _step(page):
    await wait_for_gui_entries(
        page, rdmfs_folder, present_names=(rdmfs_filename,)
    )

await run_pw(_step)
assert rdmfs_container_id() == rdmfs_initial_container_id


## rdmfs 作成ファイルの GUI メタデータとプレビューを確認する

GUI のプロパティダイアログにサイズ、対象ストレージに応じた日時・最終更新者、正しいパスが表示されること。続けてファイル詳細を開き、rdmfs から書き込んだ内容がプレビューされること。


In [ ]:
if target_storage_id == 'osfstorage':
    expected_create_time = 'nonempty'
    expected_update_time = 'nonempty'
    expected_updated_by = 'nonempty'
else:
    expected_create_time = 'empty'
    expected_update_time = 'nonempty'
    expected_updated_by = 'empty'

async def _step(page):
    await grdm.get_select_file_extension_locator(page, rdmfs_filename).click()
    await page.locator(
        '//i[contains(@class, "fa-info-circle")]/../*[text() = "プロパティ"]'
    ).click()
    modal = page.locator('//*[@id = "tb-tbody"]//*[@class = "modal-content"]')
    size = modal.locator('//*[text() = "サイズ: "]/following-sibling::span')
    create_time = modal.locator('//*[text() = "作成日時: "]/following-sibling::span')
    update_time = modal.locator('//*[text() = "更新日時: "]/following-sibling::span')
    updated_by = modal.locator('//*[text() = "最終更新者: "]/following-sibling::span')
    path = modal.locator('//*[text() = "パス: "]/following-sibling::span')
    await expect(size).not_to_be_empty(timeout=transition_timeout)
    await grdm._expect_empty_or_not(create_time, expected_create_time)
    await grdm._expect_empty_or_not(update_time, expected_update_time)
    await grdm._expect_empty_or_not(updated_by, expected_updated_by)
    await expect(path).to_have_text(f'/{rdmfs_folder}/{rdmfs_filename}')
    await modal.get_by_text('閉じる', exact=True).click()

    await grdm.get_select_file_title_locator(page, rdmfs_filename).click()
    filename_body, filename_ext = os.path.splitext(rdmfs_filename)
    await expect(
        page.locator(f'//h2[contains(text(), "{filename_body}")]//*[@id = "file-ext"]')
    ).to_have_text(filename_ext, timeout=transition_timeout)
    preview = page.locator('iframe[src*="/render"]').content_frame.locator('pre')
    await expect(preview).to_have_text(rdmfs_content, timeout=transition_timeout * 5)

await run_pw(_step)


## rdmfs でファイルを改名・削除し、GUI の状態更新を確認する

同じマウントでファイルを改名し、GUI 上で旧名が消えて新名が現れることを確認する。続けてファイルとフォルダを削除し、GUI のツリーから消えること。コンテナ ID が変わっておらず、再マウントされていないことも確認する。


In [ ]:
rdmfs_renamed_filename = '日本語-rdmfs-renamed.txt'
rdmfs_renamed_path = f'{rdmfs_folder_path}/{rdmfs_renamed_filename}'

async def _step(page):
    await page.goto(f'{project_url}files/')
    await wait_for_gui_entries(
        page, rdmfs_folder, present_names=(rdmfs_filename,)
    )

await run_pw(_step)

docker_exec('mv', rdmfs_file_path, rdmfs_renamed_path)

async def _step(page):
    await wait_for_gui_entries(
        page,
        rdmfs_folder,
        present_names=(rdmfs_renamed_filename,),
        absent_names=(rdmfs_filename,),
    )

await run_pw(_step)

docker_exec('rm', rdmfs_renamed_path)
docker_exec('rmdir', rdmfs_folder_path)

async def _step(page):
    deadline = time.monotonic() + sync_timeout
    folder = grdm.get_select_folder_title_locator(page, rdmfs_folder)
    while time.monotonic() < deadline:
        await page.reload(wait_until='domcontentloaded')
        if not await folder.is_visible():
            return
        await page.wait_for_timeout(sync_poll_interval * 1000)
    raise AssertionError(f'Timed out waiting for deleted GUI folder: {rdmfs_folder}')

await run_pw(_step)
assert rdmfs_container_id() == rdmfs_initial_container_id


## rdmfs コンテナを停止する

双方向試験に使用した同一コンテナを停止・削除する。以後、Personal Access Token を使用するプロセスが残っていないこと。


In [ ]:
removed_container = run_command(
    ['docker', 'rm', '--force', rdmfs_container_name]
).stdout.strip()
assert removed_container == rdmfs_container_name


## GUI から試験対象プロジェクトを削除する

プロジェクト設定の「プロジェクトを削除」を実行し、ダッシュボードから対象プロジェクトが消えること。


In [ ]:
async def _step(page):
    await page.goto(project_url)
    await grdm.delete_project(page, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
    await expect(page.locator(
        f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]'
    )).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)


## GUI から Personal Access Token を削除する

ユーザー設定のトークン一覧で、この Notebook が作成したトークンを非アクティブ化し、一覧から消えること。


In [ ]:
async def _step(page):
    auth_toggle = page.locator('//a[@data-test-auth-dropdown-toggle]')
    await expect(auth_toggle).to_be_visible(timeout=transition_timeout)
    await auth_toggle.click()

    settings_link = page.locator('//a[@data-test-ad-settings]')
    await expect(settings_link).to_be_visible(timeout=transition_timeout)
    await settings_link.click()

    personal_tokens_link = page.locator('a[href="/settings/tokens/"]')
    await expect(personal_tokens_link).to_be_visible(timeout=transition_timeout)
    await personal_tokens_link.click()

    token_row = page.locator(f'//tr[.//span[text() = "{token_name}"]]')
    await expect(token_row).to_be_visible(timeout=transition_timeout)
    await token_row.locator(
        'xpath=.//i[contains(@class, "fa-times") and contains(@class, "text-danger")]/..'
    ).click()
    confirm_button = page.locator('//*[@data-bb-handler = "confirm"]')
    await expect(confirm_button).to_be_visible(timeout=transition_timeout)
    await confirm_button.click()
    await expect(token_row).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)
rdm_token = None


## GUI からログアウトする

ユーザーメニューからログアウトし、未ログインのトップページが表示されること。


In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)


## 終了処理を実施する

ブラウザコンテキストを終了し、一時ファイルを削除する。スクリーンショット、動画、HAR、コンソールログは結果ディレクトリに保存されること。


In [ ]:
await finish_pw_context(last_path=default_result_path)
shutil.rmtree(work_dir)
